In [ ]:
!pip install -q unsloth transformers datasets accelerate peft trl bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.


In [ ]:
!pip install -q "fsspec==2025.12.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.3.0 requires fsspec[http]<=2025.9.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.


In [ ]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)

(True, 'Tesla T4')

In [ ]:
from datasets import load_dataset

full_dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
split_dataset = full_dataset.train_test_split(test_size=100, seed=3407)

raw_train = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_example(example):
    question = example["question"]
    context = " ".join(example["context"]["contexts"])
    decision = example["final_decision"]
    long_answer = example["long_answer"]

    text = f"""### Instruction:
Answer the following medical question with YES, NO, or MAYBE, then explain briefly.

### Question:
{question}

### Context:
{context}

### Response:
Answer: {decision.upper()}
Explanation: {long_answer}"""
    return {"text": text}

dataset = raw_train.map(format_example)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])

eval_dataset.save_to_disk("pubmedqa_eval_holdout")
print(f"Training on {len(dataset)} examples, holding out {len(eval_dataset)} for evaluation")

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Training on 900 examples, holding out 100 for evaluation


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=1024,
    dtype=torch.float16,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.7 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    max_steps=600,   # was 200
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="medical-qlora-output",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/900 [00:00<?, ? examples/s]

In [8]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 6 | Total steps = 600
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,1.484518
20,1.420269
30,1.407171
40,1.384585
50,1.397789
60,1.365124
70,1.434691
80,1.467086
90,1.543159
100,1.552795


Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in medical-qlora-output/checkpoint-600/tokenizer_config.json.


TrainOutput(global_step=600, training_loss=0.8837352649370829, metrics={'train_runtime': 7600.3145, 'train_samples_per_second': 0.632, 'train_steps_per_second': 0.079, 'total_flos': 8.842640748173722e+16, 'train_loss': 0.8837352649370829, 'epoch': 5.311111111111111})

In [9]:
model.save_pretrained("medical_lora_adapter")
tokenizer.save_pretrained("medical_lora_adapter")

Unsloth: Restored added_tokens_decoder metadata in medical_lora_adapter/tokenizer_config.json.


('medical_lora_adapter/tokenizer_config.json',
 'medical_lora_adapter/tokenizer.json')

In [10]:
from google.colab import files
import shutil

shutil.make_archive("medical_lora_adapter", 'zip', "medical_lora_adapter")
files.download("medical_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
FastLanguageModel.for_inference(model)

prompt = """### Instruction:
Answer the following medical question.

### Question:
What are common symptoms of asthma?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Answer the following medical question.

### Question:
What are common symptoms of asthma?

### Response:
Answer: Cough, Dyspnea, Wheezing

### Context:
Asthma is a common chronic disease with increasing prevalence in the last decades. A definite diagnosis is based on a combination of symptoms, physical examination, laboratory and functional findings consistent with asthma. The assessment of symptoms by patients is of crucial importance in the diagnosis of asthma. To investigate the frequency of symptoms of asthma in subjects who received a diagnosis of asthma by a physician in Spain (ASPHERE project). Cross-sectional study in adults (18<or=30, 30<or=65 years) with a diagnosis of asthma made by a physician in the past. The study was carried out in 13 Spanish cities. Patients with a diagnosis of asthma by an HLA I/DD. The instrument used was the International Study of Asthma Symptoms (ISAAC). The frequency and the percentage of symptoms were analyzed. A total of 1530 pat

In [13]:
from datasets import load_from_disk

eval_dataset = load_from_disk("pubmedqa_eval_holdout")

def build_prompt(example):
    question = example["question"]
    context = " ".join(example["context"]["contexts"])
    return f"""### Instruction:
Answer the following medical question with YES, NO, or MAYBE, then explain briefly.

### Question:
{question}

### Context:
{context}

### Response:
"""

def get_answer(m, tok, prompt):
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    outputs = m.generate(**inputs, max_new_tokens=100, temperature=0.3, top_p=0.9)
    text = tok.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

def extract_label_v3(text):
    t = text.lower()
    if "answer: yes" in t: return "yes"
    if "answer: no" in t: return "no"
    if "answer: maybe" in t: return "maybe"
    return "unclear"

FastLanguageModel.for_inference(model)

ft_correct = 0
results = []
for ex in eval_dataset:
    prompt = build_prompt(ex)
    raw = get_answer(model, tokenizer, prompt)
    pred = extract_label_v3(raw)          # <-- fixed: was extract_label
    correct = (pred == ex["final_decision"])
    ft_correct += correct
    results.append({"question": ex["question"], "gold": ex["final_decision"], "ft_pred": pred, "raw": raw})

print(f"Fine-tuned model: {ft_correct}/{len(eval_dataset)} correct")

Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Fine-tuned model: 74/100 correct


In [14]:
del model
torch.cuda.empty_cache()

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=1024,
    dtype=torch.float16,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_correct = 0
for i, ex in enumerate(eval_dataset):
    prompt = build_prompt(ex)
    raw = get_answer(base_model, tokenizer, prompt)
    pred = extract_label_v3(raw)
    correct = (pred == ex["final_decision"])
    base_correct += correct
    results[i]["base_pred"] = pred
    results[i]["base_raw"] = raw

print(f"Base model: {base_correct}/{len(eval_dataset)} correct")
print(f"Fine-tuned model: {ft_correct}/{len(eval_dataset)} correct")

==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https

Base model: 0/100 correct
Fine-tuned model: 74/100 correct


In [15]:
for r in results[:5]:
    print("GOLD:", r["gold"])
    print("BASE RAW:", r["base_raw"][:150])
    print("-" * 50)

GOLD: yes
BASE RAW: YES. The year of RP is a predictor of outcome in prostate cancer. This is because the year of RP is associated with the year of diagnosis. The year of
--------------------------------------------------
GOLD: maybe
BASE RAW: YES. The use of hydrophilic guidewires has increased the technical success rate of peripheral PTA. The use of hydrophilic guidewires has increased the
--------------------------------------------------
GOLD: yes
BASE RAW: S

### Explanation:
The purpose of this study was to evaluate the clinical usefulness of a fetal anatomic survey on follow-up antepartum sonograms. A 
--------------------------------------------------
GOLD: yes
BASE RAW: MAYBE. The study was conducted in an outpatient stroke centre. The participants were individuals post first stroke who acknowledged walking slower tha
--------------------------------------------------
GOLD: maybe
BASE RAW: MAYBE. The question is not clear. It is not clear if the question is asking if the medic